In [7]:
# train_gat_node_fixed.py
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

NODE_CSV  = "GNNDatasets/node.csv"
EDGE_CSV  = "GNNDatasets/node_edges.csv"
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# ----------------------------- Load nodes -----------------------------
nodes_df = pd.read_csv(NODE_CSV)

label_col = None
for cand in ["label", "is_trojan", "trojan", "target"]:
    if cand in nodes_df.columns:
        label_col = cand; break
if label_col is None:
    nodes_df["label"] = nodes_df["circuit_name"].astype(str).str.contains("__trojan_").astype(int)
    label_col = "label"

nodes_df["uid"] = nodes_df["circuit_name"].astype(str) + "::" + nodes_df["node"].astype(str)

feat_df = nodes_df.copy()
if "gate_type" in feat_df.columns:
    gate_oh = pd.get_dummies(feat_df["gate_type"], prefix="gt")
    feat_df = pd.concat([feat_df.drop(columns=["gate_type"]), gate_oh], axis=1)

exclude = {"uid","node","circuit_name",label_col}
num_cols = [c for c in feat_df.columns if c not in exclude and pd.api.types.is_numeric_dtype(feat_df[c])]
X = feat_df[num_cols].fillna(0.0).values.astype(np.float32)
y = nodes_df[label_col].values.astype(np.int64)

# ----------------------------- Load edges; add missing nodes -----------------------------
edges_df = pd.read_csv(EDGE_CSV)
edges_df["src_uid"] = edges_df["circuit_name"].astype(str) + "::" + edges_df["src"].astype(str)
edges_df["dst_uid"] = edges_df["circuit_name"].astype(str) + "::" + edges_df["dst"].astype(str)

known_uids = set(nodes_df["uid"])
edge_uids = set(edges_df["src_uid"]).union(set(edges_df["dst_uid"]))
missing = list(edge_uids - known_uids)

if missing:
    zero_row = np.zeros((1, X.shape[1]), dtype=np.float32)
    addX = np.repeat(zero_row, len(missing), axis=0)
    addY = -1*np.ones(len(missing), dtype=np.int64)
    add_df = pd.DataFrame({
        "uid": missing,
        "circuit_name": [u.split("::",1)[0] for u in missing],
        "node": [u.split("::",1)[1] for u in missing],
        label_col: addY
    })
    X = np.vstack([X, addX])
    y = np.concatenate([y, addY])
    nodes_df = pd.concat([nodes_df, add_df], ignore_index=True)

uid_to_idx = {u:i for i,u in enumerate(nodes_df["uid"].tolist())}
src_idx = edges_df["src_uid"].map(uid_to_idx).dropna().astype(int).values
dst_idx = edges_df["dst_uid"].map(uid_to_idx).dropna().astype(int).values
edge_index = np.stack([np.concatenate([src_idx, dst_idx]),
                       np.concatenate([dst_idx, src_idx])], axis=0)

# ----------------------------- Scale features -----------------------------
labeled_mask_np = (y >= 0)
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[labeled_mask_np] = scaler.fit_transform(X_scaled[labeled_mask_np])
if (~labeled_mask_np).any():
    X_scaled[~labeled_mask_np] = (X_scaled[~labeled_mask_np] - scaler.mean_) / np.sqrt(scaler.var_ + 1e-8)

# ----------------------------- Splits -----------------------------
idx_all = np.where(labeled_mask_np)[0]
y_all = y[labeled_mask_np]

idx_train, idx_tmp, y_train, y_tmp = train_test_split(
    idx_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)

# ----------------------------- Torch tensors -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_t = torch.from_numpy(X_scaled).to(device)
y_t = torch.from_numpy(y).to(device)
edge_index_t = torch.from_numpy(edge_index).long().to(device)

train_mask_t = torch.zeros(len(y), dtype=torch.bool, device=device); train_mask_t[idx_train] = True
val_mask_t   = torch.zeros(len(y), dtype=torch.bool, device=device); val_mask_t[idx_val]   = True
test_mask_t  = torch.zeros(len(y), dtype=torch.bool, device=device); test_mask_t[idx_test]  = True
labeled_mask_t = torch.from_numpy(labeled_mask_np).to(device)

# ========================================================================
#                    Improved Graph Attention Network
# ========================================================================

# robust scatter helpers (no torch_scatter dependency)
def scatter_add(src, index, dim_size=None):
    """
    Accumulate src values into an output using index, like torch_scatter.scatter_add.
    If dim_size is None the function infers it from index.
    src: 1D or 2D tensor where first dim == len(index) or shape[0] == len(index)
    index: 1D long tensor of target positions (same length as number of rows in src)
    """
    if index.numel() == 0:
        if src.dim() == 1:
            return torch.zeros((0,), device=src.device, dtype=src.dtype)
        else:
            return torch.zeros((0, src.size(1)), device=src.device, dtype=src.dtype)

    if dim_size is None:
        dim_size = int(index.max().item()) + 1
    # if src is vector apply index_add_ on 1D; if 2D do index_add_ with broadcasting
    if src.dim() == 1:
        out = torch.zeros(dim_size, device=src.device, dtype=src.dtype)
        out.index_add_(0, index, src)
    else:
        out = torch.zeros(dim_size, src.size(1), device=src.device, dtype=src.dtype)
        # for index_add_ with 2D, do index_add per column using view
        out.index_add_(0, index, src)
    return out

def scatter_softmax(src, index, dim_size=None):
    """
    Numerically-stable softmax grouped by `index` (like scatter_softmax(dst) grouping).
    src: 1D tensor of size E (edge scores)
    index: 1D long tensor of size E (destination node for each edge)
    """
    if index.numel() == 0:
        return src.new_empty(0)

    if dim_size is None:
        dim_size = int(index.max().item()) + 1

    # compute max per index for numerical stability
    # initialize with -inf so positions without contributions don't interfere
    max_per_index = torch.full((dim_size,), -float('inf'), device=src.device, dtype=src.dtype)
    # put max values (for repeated indices the last write is not guaranteed; we'll use scatter approach)
    # to compute correctly, build per-index maximum manually:
    # create a tensor of shape (E,) for placement: we want max over entries grouped by index
    # approach: compute per-index max by iterating via scatter_add of boolean masks? simpler: use segment trick:
    # we'll compute max using a loop over unique indices when data is small-ish (safe), but faster method:
    # use torch_scatter normally - but here we mimic with a reduction:
    # Efficient vectorized method: use scatter_add on exp but for max we can compute using indexing:
    # create a tensor filled with -inf and then for each edge assign max via advanced indexing:
    # get per-index max via scatter-like pythonic:
    # compute per-index max using scatter via grouping by index:
    # Create a buffer of lists is not vectorized; but we can use torch.zeros and scatter_max isn't available.
    # Use torch.segment_max-like trick: use scatter_add on one-hot weighted by src? Simpler approach below:
    # Use torch.zeros to hold -inf and then do elementwise maximum via index loop. This is still fast enough in pure python for moderate E.
    max_per_index_cpu = max_per_index
    # We'll use a vectorized trick: for each unique index compute max using boolean mask (this is okay for moderate-size graphs).
    unique_idx = torch.unique(index)
    for u in unique_idx:
        mask = (index == u)
        max_per_index_cpu[u] = torch.max(src[mask])
    max_per_index = max_per_index_cpu

    max_per_edge = max_per_index[index]
    exp_scores = torch.exp(src - max_per_edge)
    denom = scatter_add(exp_scores, index, dim_size=dim_size)
    out = exp_scores / (denom[index] + 1e-16)
    return out

class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.0, alpha=0.2, residual=True, layer_norm=True):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        # keep a_src / a_dst shaped (out_dim, 1) to reuse your previous arithmetic
        self.a_src = nn.Parameter(torch.empty(out_dim, 1))
        self.a_dst = nn.Parameter(torch.empty(out_dim, 1))
        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)
        self.residual = residual and (in_dim == out_dim)
        self.layer_norm = nn.LayerNorm(out_dim) if layer_norm else None

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)

    def forward(self, x, edge_index):
        # x: [N, in_dim]
        # edge_index: [2, E] with src, dst
        h = self.W(x)                      # [N, out_dim]
        src, dst = edge_index              # each length E (long)
        # attention scalar per node (before combining)
        alpha_src = (h @ self.a_src).squeeze(-1)   # [N]
        alpha_dst = (h @ self.a_dst).squeeze(-1)   # [N]
        e = self.leakyrelu(alpha_src[src] + alpha_dst[dst])  # [E]

        # normalize attention per destination node (softmax over incoming edges)
        alpha = scatter_softmax(e, dst)   # [E]

        # message aggregation: out[v] = sum_{u->v} alpha_{uv} * h[u]
        out = torch.zeros_like(h)
        out.index_add_(0, dst, h[src] * alpha.unsqueeze(-1))

        if self.layer_norm is not None:
            out = self.layer_norm(out)

        if self.residual:
            out = out + x

        return self.dropout(out)

class GAT(nn.Module):
    def __init__(self, in_dim, hid_dim=96, out_dim=2, dropout=0.35, alpha=0.2):
        super().__init__()
        self.g1 = GATLayer(in_dim, hid_dim, dropout=dropout, alpha=alpha)
        self.g2 = GATLayer(hid_dim, out_dim, dropout=dropout, alpha=alpha, residual=False)
        self.do = nn.Dropout(dropout)
        self.batch_norm = nn.BatchNorm1d(hid_dim)

    def forward(self, x, edge_index):
        x = self.g1(x, edge_index)
        x = self.batch_norm(x)
        x = F.elu(x)
        x = self.do(x)
        x = self.g2(x, edge_index)
        return x

# instantiate model
model = GAT(in_dim=X_t.size(1), hid_dim=182, out_dim=2, dropout=0.15).to(device)

# ----------------------------- Loss, optimizer -----------------------------
train_labels = y_t[train_mask_t]
classes, counts = torch.unique(train_labels, return_counts=True)
num_pos = counts[classes==1].item() if (classes==1).any() else 1
num_neg = counts[classes==0].item() if (classes==0).any() else 1
weight_pos = (num_neg + num_pos) / (2.0 * num_pos)
weight_neg = (num_neg + num_pos) / (2.0 * num_neg)
class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=5e-4)

# ----------------------------- Training -----------------------------
def evaluate(mask_t):
    model.eval()
    with torch.no_grad():
        logits = model(X_t, edge_index_t)
        pred = logits.argmax(dim=1)
        msk = mask_t & (y_t >= 0)
        if msk.sum() == 0: return 0.0
        return (pred[msk] == y_t[msk]).float().mean().item()

best_val, best_state = -1.0, None
patience, patience_cnt = 20, 0
EPOCHS = 100

for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad()
    logits = model(X_t, edge_index_t)
    loss = criterion(logits[train_mask_t], y_t[train_mask_t])
    # guard against NaN loss (just in case)
    if torch.isnan(loss):
        print(f"Epoch {epoch:03d} | Loss is NaN -> aborting epoch")
        break
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 2.0)
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        val_acc = evaluate(val_mask_t)
        test_acc = evaluate(test_mask_t)
        print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | Val {val_acc:.4f} | Test {test_acc:.4f}")
        if val_acc > best_val + 1e-4:
            best_val = val_acc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print("Early stopping."); break

if best_state is not None:
    model.load_state_dict(best_state)

# ----------------------------- Final eval -----------------------------
model.eval()
with torch.no_grad():
    logits = model(X_t, edge_index_t)
    preds = logits.argmax(dim=1)

msk = (test_mask_t & (y_t >= 0)).cpu().numpy()
y_true = y_t.cpu().numpy()[msk]
y_pred = preds.cpu().numpy()[msk]

acc = (y_true == y_pred).mean()
print("\nFinal Evaluation (Node-Level)")
print("=============================")
print(f"Test Accuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, labels=[0,1], target_names=["clean","trojan"], digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0,1]))


Epoch 001 | Loss 1.1780 | Val 0.0547 | Test 0.0540
Epoch 010 | Loss 0.4363 | Val 0.9952 | Test 0.9952
Epoch 020 | Loss 0.4226 | Val 0.9957 | Test 0.9958
Epoch 030 | Loss 0.4128 | Val 0.9958 | Test 0.9959
Epoch 040 | Loss 0.4039 | Val 0.9958 | Test 0.9960
Epoch 050 | Loss 0.3953 | Val 0.9958 | Test 0.9961
Epoch 060 | Loss 0.3875 | Val 0.9959 | Test 0.9961
Epoch 070 | Loss 0.3799 | Val 0.9959 | Test 0.9961
Epoch 080 | Loss 0.3721 | Val 0.9959 | Test 0.9961
Epoch 090 | Loss 0.3647 | Val 0.9960 | Test 0.9961
Epoch 100 | Loss 0.3572 | Val 0.9960 | Test 0.9961

Final Evaluation (Node-Level)
Test Accuracy: 0.9961

Classification Report:
              precision    recall  f1-score   support

       clean     1.0000    0.9845    0.9922      9159
      trojan     0.9949    1.0000    0.9974     27556

    accuracy                         0.9961     36715
   macro avg     0.9974    0.9922    0.9948     36715
weighted avg     0.9962    0.9961    0.9961     36715

Confusion Matrix:
[[ 9017   142]
 [

In [ ]:
# ==================== CELL 3: GBLA (Trojan detection) - no defense + five defenses ====================
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ---------- Attack hyperparameters ----------
ATTACK_ITERATIONS = 100
ATTACKS_PER_CLASS  = 20
DUMMY_LR          = 0.1
L2_REG            = 0.001
RANDOM_SEED       = 42
SAVE_CSV          = "gat_gbla_trojan_defenses_results.csv"
SAVE_PDF          = "gat_gbla_trojan_defenses_plots.pdf"

# ---------- reproducibility ----------
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ---------- sanity checks ----------
required = ["model", "X_t", "y_t", "edge_index_t", "train_mask_t"]
missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(f"Missing required vars (run cell1): {missing}")

device = next(model.parameters()).device
model.eval()
for p in model.parameters():
    p.requires_grad = True

# ---------- prepare lists ----------
train_idx_all = torch.nonzero(train_mask_t & (y_t >= 0), as_tuple=False).view(-1).cpu().numpy().tolist()
train_labels = y_t[train_mask_t].cpu().numpy()
classes_present = sorted(np.unique(train_labels).tolist())          # should be [0,1]
rng = np.random.default_rng(RANDOM_SEED)
features_orig = X_t.clone().detach()

# ---------- defenses ----------
def apply_defense_to_tensor(tensor, defense):
    if defense == "dp":
        return tensor + 0.02 * torch.randn_like(tensor)
    if defense == "clip":
        return torch.clamp(tensor, -0.5, 0.5)
    if defense == "secure":
        return torch.zeros_like(tensor)
    if defense == "quant":
        return torch.round(tensor * 10.0) / 10.0
    if defense == "advtrain":
        return tensor + 0.01 * torch.sign(torch.randn_like(tensor))
    return tensor

def apply_defense_to_grads(grads, defense):
    # grads is list of tensors (no None because we collect zeros-like below)
    return [apply_defense_to_tensor(g, defense).detach() for g in grads]

defenses = ["none", "dp", "clip", "secure", "quant", "advtrain"]
results = []

print("Starting GBLA (trojan detection) over defenses...")

# ---------- main loop ----------
for defense in defenses:
    print(f"\n=== Defense: {defense.upper()} ===")
    for cls in classes_present:
        # candidate indices drawn from training set and matching class
        cand = [int(i) for i in train_idx_all if int(y_t[i].item()) == int(cls)]
        if len(cand) == 0:
            print(f" - class {cls} has no train candidates, skipping")
            continue
        if len(cand) <= ATTACKS_PER_CLASS:
            target_indices = cand
        else:
            target_indices = list(rng.choice(cand, size=ATTACKS_PER_CLASS, replace=False))

        print(f"Class {cls}: attacking {len(target_indices)} nodes")
        for target_idx in tqdm(target_indices, desc=f"{defense} | class {cls}", leave=False):
            model.zero_grad()

            # --- Step 1: compute true gradients for this target ---
            logits = model(features_orig.to(device), edge_index_t)
            tgt_label = y_t[target_idx:target_idx+1].to(device)
            loss_t = nn.CrossEntropyLoss()(logits[target_idx:target_idx+1], tgt_label)
            loss_t.backward()

            # collect target gradients (replace None with zeros_like param)
            target_grads = []
            for p in model.parameters():
                if p.grad is None:
                    target_grads.append(torch.zeros_like(p))
                else:
                    target_grads.append(p.grad.detach().clone())

            # apply defense (if none -> unchanged)
            grads_used = target_grads if defense == "none" else apply_defense_to_grads(target_grads, defense)

            # --- Step 2: reconstruct by optimizing dummy feature for this node ---
            dummy_feat = torch.randn_like(features_orig[target_idx:target_idx+1], device=device, requires_grad=True)
            dummy_opt = optim.Adam([dummy_feat], lr=DUMMY_LR)

            for it in range(ATTACK_ITERATIONS):
                dummy_features = features_orig.clone().to(device)
                dummy_features[target_idx:target_idx+1] = dummy_feat

                model.zero_grad()
                dummy_logits = model(dummy_features, edge_index_t)
                dummy_loss_single = nn.CrossEntropyLoss()(dummy_logits[target_idx:target_idx+1], tgt_label)

                # grads wrt model params for dummy loss (create_graph so we can differentiate)
                dummy_grads = torch.autograd.grad(dummy_loss_single, tuple(model.parameters()),
                                                  create_graph=True, allow_unused=True)

                # compute squared difference between dummy_grads and grads_used
                grad_diff = torch.tensor(0.0, device=device)
                for dg, tg in zip(dummy_grads, grads_used):
                    if dg is None:
                        dg = torch.zeros_like(tg, device=device)
                    # tg is a tensor (we ensured earlier)
                    grad_diff = grad_diff + torch.sum((dg - tg) ** 2)

                reg = torch.norm(dummy_feat, p=2)
                total = grad_diff + L2_REG * reg

                dummy_opt.zero_grad()
                total.backward()
                dummy_opt.step()

            # --- Step 3: compute metrics ---
            orig_feat = features_orig[target_idx:target_idx+1].detach().cpu()
            recon_feat = dummy_feat.detach().cpu()

            abs_l2 = torch.norm(recon_feat - orig_feat, p=2).item()
            rel_l2 = abs_l2 / (torch.norm(orig_feat, p=2).item() + 1e-8)
            cos_sim = F.cosine_similarity(recon_feat, orig_feat).mean().item()

            results.append({
                "defense": defense,
                "class": int(cls),
                "node_idx": int(target_idx),
                "abs_l2": float(abs_l2),
                "rel_l2": float(rel_l2),
                "cos_sim": float(cos_sim)
            })

# ---------- save results ----------
df = pd.DataFrame(results)
df.to_csv(SAVE_CSV, index=False)
print(f"\nSaved results -> {SAVE_CSV} (rows={len(df)})\n")

# ---------- print summaries ----------
print("Overall per-defense summary (mean ± std):")
print(df.groupby("defense")[["abs_l2","rel_l2","cos_sim"]].agg(["mean","std","count"]))

print("\nPer-defense per-class summary (mean ± std):")
per_class = df.groupby(["defense","class"])[["abs_l2","rel_l2","cos_sim"]].agg(["mean","std","count"])
print(per_class)

# ---------- plotting ----------
plt.rcParams.update({'font.size': 10})
n_def = len(defenses)
width = 0.28        # width for each class box within a defense group
centers = np.arange(n_def)

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

metric_names = ["abs_l2","rel_l2","cos_sim"]
y_ticks = None

colors = ["#4C72B0", "#DD8452"]  # blue=clean, orange=trojan

for ax, metric in zip(axes, metric_names):
    # build data arrays grouped as [for each defense -> [class0_samples, class1_samples]]
    grouped = []
    for d in defenses:
        group0 = df[(df["defense"]==d) & (df["class"]==0)][metric].values
        group1 = df[(df["defense"]==d) & (df["class"]==1)][metric].values
        grouped.append((group0, group1))

    # draw boxplots: for each defense, plot two thin boxes offset left/right
    positions_left  = centers - width/2.0
    positions_right = centers + width/2.0
    data_left  = [g[0] for g in grouped]
    data_right = [g[1] for g in grouped]

    bp_l = ax.boxplot(data_left, positions=positions_left, widths=width*0.9, patch_artist=True,
                      showfliers=False, manage_ticks=False)
    bp_r = ax.boxplot(data_right, positions=positions_right, widths=width*0.9, patch_artist=True,
                      showfliers=False, manage_ticks=False)

    for patch in bp_l['boxes']:
        patch.set_facecolor(colors[0])
        patch.set_alpha(0.7)
    for patch in bp_r['boxes']:
        patch.set_facecolor(colors[1])
        patch.set_alpha(0.7)

    # style medians/whiskers
    for key in ('medians','whiskers','caps'):
        for el in bp_l.get(key, []):
            el.set_color('k'); el.set_linewidth(0.6)
        for el in bp_r.get(key, []):
            el.set_color('k'); el.set_linewidth(0.6)

    ax.set_ylabel(metric)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # set sensible ticks for abs_l2 (if range large) and rel_l2
    if metric == "abs_l2":
        ax.set_yticks([0,5,10,15,20,25])
    if metric == "rel_l2":
        ax.set_yticks([0,1,2,3,4])

# x labels and shared legend below plots
axes[-1].set_xticks(centers)
axes[-1].set_xticklabels(defenses, rotation=30, ha='right')
handles = [plt.Rectangle((0,0),1,1, facecolor=c, alpha=0.7) for c in colors]
fig.legend(handles, ["clean (class 0)","trojan (class 1)"], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.03))

plt.tight_layout(rect=[0,0.04,1,1])
fig.savefig(SAVE_PDF, bbox_inches='tight', dpi=200)
print(f"Saved figure -> {SAVE_PDF}")
plt.show()


Starting GBLA (trojan detection) over defenses...

=== Defense: NONE ===
Class 0: attacking 20 nodes


none | class 0:   0%|                                                                                                                                                                                | 0/20 [00:00<?, ?it/s]